<a href="https://colab.research.google.com/github/vectara/example-notebooks/blob/main/notebooks/api-examples/10-structured-output-multi-step.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Structured Output and Multi-Step Agents

This notebook demonstrates two powerful Vectara agent features:

1. **Structured Output**: Configure agents to return validated JSON conforming to a schema, using the model's native structured outputs capability
2. **Multi-Step Agents**: Define agents with multiple steps and conditional routing, enabling workflow-like behavior

You'll learn how to:
1. Create agents with structured output parsers for guaranteed JSON responses
2. Define multi-step agents with conditional transitions between steps
3. Use step-specific tool restrictions with `allowed_tools`
4. Parse structured output events and step transition events
5. Combine both features to build a classifier-router pattern

> **Related: see notebook 14 for sequential pipelines, conditional gates, and `reentry_step`.** This notebook covers the **classifier-router fan-out** pattern — one classifier branches to one of N terminal handlers. Notebook 14 covers the complementary **plan-then-execute pipeline** (each phase chains forward), conditional gating that skips heavy phases, and `reentry_step` for multi-turn follow-up flows. Read both for the full step-orchestration picture.

## About Vectara

[Vectara](https://vectara.com/) is an Agent Platform for trusted enterprise AI — a unified Agentic RAG platform with built-in retrieval, orchestration, and governance. See [Notebook 1](1-corpus-creation.ipynb) for the full overview of features and deployment options.

## Getting Started

This notebook assumes you've completed Notebooks 1 and 2:
- Notebook 1: Created two corpora (ai-research-papers and vectara-docs) with Boomerang embeddings
- Notebook 2: Ingested AI research papers and Vectara documentation

## Setup

In [1]:
import os
import requests
import json
from datetime import datetime

api_key = os.environ['VECTARA_API_KEY']

research_corpus_key = 'tutorial-ai-research-papers'
docs_corpus_key = 'tutorial-vectara-docs'

BASE_URL = "https://api.vectara.io/v2"

headers = {
    "x-api-key": api_key,
    "Content-Type": "application/json"
}

print("Setup complete.")

Setup complete.


In [2]:
# Load the shared helpers (delete_and_create_agent).
# vectara_utils.py sits next to this notebook in the repo; Colab fetches it on
# first run so the same notebook works locally and in a fresh Colab runtime.
try:
    from vectara_utils import delete_and_create_agent
except ImportError:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/vectara/example-notebooks/main/notebooks/api-examples/vectara_utils.py",
        "vectara_utils.py",
    )
    from vectara_utils import delete_and_create_agent

import vectara_utils
vectara_utils.configure(BASE_URL, headers)


In [3]:
def chat_with_agent(agent_key, session_key, message, show_events=False):
    """Send a message to an agent and return its terminal output.

    For a single-step agent with a structured output parser, this returns
    the validated dict from `structured_output`. For a multi-step agent
    that transitions classifier -> handler, we want the handler's final
    `agent_output`, not the classifier's intermediate `structured_output`.
    We walk every event and keep the last terminal output we saw, which
    naturally picks the right one in both cases.
    """
    message_data = {
        "messages": [{"type": "text", "content": message}],
        "stream_response": False
    }
    url = f"{BASE_URL}/agents/{agent_key}/sessions/{session_key}/events"
    response = requests.post(url, headers=headers, json=message_data)

    if response.status_code != 201:
        print(f"Error: {response.status_code} - {response.text}")
        return None

    event_data = response.json()
    if show_events:
        print("\n--- Agent Events ---")
        for event in event_data.get('events', []):
            event_type = event.get('type', 'unknown')
            if event_type == 'step_transition':
                print(f"  Step transition: {event.get('from_step', '?')} -> {event.get('to_step', '?')}")
            elif event_type == 'structured_output':
                print(f"  Structured output ({event.get('schema_name', '?')}): "
                      f"{json.dumps(event.get('content', {}))[:200]}")
            elif event_type == 'tool_input':
                print(f"  Tool call: {event.get('tool_configuration_name', '?')}")
            elif event_type == 'tool_output':
                print(f"  Tool response: {event.get('tool_configuration_name', '?')}")
            elif event_type == 'agent_output':
                print(f"  Agent output: {event.get('content', '')[:200]}...")
        print("---\n")

    last_output = None
    for event in event_data.get('events', []):
        etype = event.get('type')
        if etype == 'structured_output':
            last_output = event.get('content')
        elif etype == 'agent_output':
            last_output = event.get('content', 'No content')
    return last_output if last_output is not None else "No output found"


## Part 1: Structured Output

### Step 1: Create an Agent with Structured Output

Instead of free-form text, you can configure an agent to return validated JSON conforming to a schema.
The `StructuredOutputParser` uses the model's native structured outputs capability to guarantee
valid JSON on every response. This is useful for entity extraction, classification, and downstream
API integrations where you need a predictable response format.

When configured, the agent emits `structured_output` events (instead of `agent_output`) containing
the validated JSON and the schema name.

In [4]:
extraction_agent_config = {
    "name": "Research Entity Extractor",
    "description": "Extracts structured entities from research questions",
    "model": {"name": "gpt-4o"},
    "first_step_name": "extract",
    "steps": {
        "extract": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "extraction_instructions",
                    "template": (
                        "You are a research entity extraction agent. Given a user question about AI research, "
                        "search the corpus for relevant information, then extract structured entities from both "
                        "the question and retrieved content.\n\n"
                        "Identify:\n"
                        "- The main research topics mentioned\n"
                        "- Specific techniques or methods referenced\n"
                        "- A brief answer to the question based on retrieved content\n"
                        "- A confidence level for your answer"
                    )
                }
            ],
            "output_parser": {
                "type": "structured",
                "json_schema": {
                    "name": "research_extraction",
                    "description": "Structured extraction of research entities and answer",
                    "strict": True,
                    "schema": {
                        "type": "object",
                        "properties": {
                            "topics": {
                                "type": "array",
                                "items": {"type": "string"}
                            },
                            "techniques": {
                                "type": "array",
                                "items": {"type": "string"}
                            },
                            "answer": {"type": "string"},
                            "confidence": {
                                "type": "string",
                                "enum": ["high", "medium", "low"]
                            }
                        },
                        "required": ["topics", "techniques", "answer", "confidence"],
                        "additionalProperties": False
                    }
                }
            }
        }
    },
    "tool_configurations": {
        "research_search": {
            "type": "corpora_search",
            "query_configuration": {
                "search": {
                    "corpora": [{"corpus_key": research_corpus_key}],
                    "limit": 10
                }
            }
        }
    }
}

extraction_agent_key = delete_and_create_agent(extraction_agent_config, "Research Entity Extractor")

Created agent 'Research Entity Extractor' (key: agt_research_entity_extractor_980c)


### Step 2: Test Structured Output

Let's send queries and verify the agent returns validated JSON matching our schema every time.

In [5]:
session_config = {"name": f"Extraction Demo {datetime.now().strftime('%Y%m%d-%H%M%S')}"}
response = requests.post(
    f"{BASE_URL}/agents/{extraction_agent_key}/sessions",
    headers=headers,
    json=session_config
)

extraction_session_key = None
if response.status_code == 201:
    extraction_session_key = response.json()["key"]
    print(f"Session created: {extraction_session_key}")
else:
    print(f"Error creating session: {response.text}")

Session created: ase_extraction_demo_20261006-_447db14a15fcf9a82846


In [6]:
result = chat_with_agent(
    extraction_agent_key,
    extraction_session_key,
    "What techniques does RAG use to reduce hallucinations in language models?",
    show_events=True
)

print("Structured Result:")
print(json.dumps(result, indent=2))

# Access fields programmatically
if isinstance(result, dict):
    print(f"\nTopics: {result.get('topics', [])}")
    print(f"Techniques: {result.get('techniques', [])}")
    print(f"Confidence: {result.get('confidence', 'N/A')}")


--- Agent Events ---
  Tool call: research_search
  Tool response: research_search
  Structured output (research_extraction): {"topics": ["Retrieval-Augmented Generation (RAG)", "hallucinations in language models", "knowledge-intensive NLP tasks"], "techniques": ["retrieval mechanisms", "fine-tuning", "self-reflection", "sel
---

Structured Result:
{
  "topics": [
    "Retrieval-Augmented Generation (RAG)",
    "hallucinations in language models",
    "knowledge-intensive NLP tasks"
  ],
  "techniques": [
    "retrieval mechanisms",
    "fine-tuning",
    "self-reflection",
    "self-consistency"
  ],
  "answer": "RAG (Retrieval-Augmented Generation) reduces hallucinations in language models by combining pre-trained parametric and non-parametric memory for language generation. This approach makes use of retrieval mechanisms to enhance the factual correctness of the generated text. By retrieving relevant information and incorporating it into the generation process, RAG models tend to p

In [7]:
# Second query — same schema, different content
result2 = chat_with_agent(
    extraction_agent_key,
    extraction_session_key,
    "How do dense retrieval models compare to sparse retrieval methods like BM25?",
    show_events=True
)

print("Structured Result:")
print(json.dumps(result2, indent=2))


--- Agent Events ---
  Tool call: research_search
  Tool response: research_search
  Structured output (research_extraction): {"topics": ["dense retrieval", "sparse retrieval", "information retrieval"], "techniques": ["BM25", "dense passage retrieval (DPR)", "dense retrieval models", "sparse models including DeepCT and SPART
---

Structured Result:
{
  "topics": [
    "dense retrieval",
    "sparse retrieval",
    "information retrieval"
  ],
  "techniques": [
    "BM25",
    "dense passage retrieval (DPR)",
    "dense retrieval models",
    "sparse models including DeepCT and SPARTA",
    "re-ranking methods"
  ],
  "answer": "Dense retrieval models, such as Dense Passage Retrieval (DPR) and ANCE, utilize neural networks to learn dense vector representations for queries and documents, aiding in more semantic retrieval processes. In contrast, sparse retrieval methods like BM25 rely on keyword matching and inverse document frequency to identify relevant documents. While dense retrieval

## Part 2: Multi-Step Agent with Classifier-Router Pattern

### Step 3: Create a Multi-Step Agent

A common production pattern is the **classifier-router**: a first step classifies the user's
intent using structured output, then conditional routing sends the conversation to a specialized
handler step. Each step can have its own instructions and restricted tool access via `allowed_tools`.

Key concepts:
- **`steps` map**: Define named steps (replaces the deprecated `first_step` field)
- **`first_step_name`**: References the entry point step in the map
- **`next_steps`**: Conditional transitions using UserFn expressions (`get('$.output.field') == 'value'`)
- **`allowed_tools`**: Restrict which tools each step can use (empty list = no tools)
- **`reentry_step`**: The step the *next* user message in the session enters. It defaults to the step the turn ended in, so each handler sets `reentry_step: "classifier"` — otherwise a second question in the same session would skip classification and go straight to the previous handler

In [8]:
classifier_agent_config = {
    "name": "Research Assistant Router",
    "description": "Multi-step agent that classifies queries and routes to specialized handlers",
    "model": {"name": "gpt-4o"},
    "first_step_name": "classifier",
    "steps": {
        "classifier": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "classifier_instructions",
                    "template": (
                        "You are a query classifier. Analyze the user's question and classify it into one of "
                        "these intents:\n\n"
                        '- "research": Questions about academic research, theoretical concepts, algorithms, or '
                        "papers (e.g., 'How does attention work in transformers?')\n"
                        '- "implementation": Questions about using Vectara APIs, SDKs, configuration, or building '
                        "applications (e.g., 'How do I set up hybrid search?')\n"
                        '- "comparison": Questions that compare, contrast, or evaluate multiple approaches '
                        "(e.g., 'BM25 vs dense retrieval?')\n\n"
                        "Classify based on the primary intent. Do not use any tools."
                    )
                }
            ],
            "output_parser": {
                "type": "structured",
                "json_schema": {
                    "name": "intent_classification",
                    "description": "Classifies user query intent for routing",
                    "strict": True,
                    "schema": {
                        "type": "object",
                        "properties": {
                            "intent": {
                                "type": "string",
                                "enum": ["research", "implementation", "comparison"]
                            },
                            "reasoning": {"type": "string"}
                        },
                        "required": ["intent", "reasoning"],
                        "additionalProperties": False
                    }
                }
            },
            "allowed_tools": [],
            "next_steps": [
                {
                    "condition": "get('$.output.intent') == 'research'",
                    "step_name": "research_handler"
                },
                {
                    "condition": "get('$.output.intent') == 'implementation'",
                    "step_name": "implementation_handler"
                },
                {
                    "step_name": "comparison_handler"
                }
            ]
        },
        "research_handler": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "research_instructions",
                    "template": (
                        "You are a research expert. The user's question has been classified as research-oriented. "
                        "Search the research papers corpus for relevant academic content and provide a thorough, "
                        "well-cited answer grounded in the retrieved papers. Focus on theoretical foundations, "
                        "key findings, and methodology."
                    )
                }
            ],
            "output_parser": {"type": "default"},
            "allowed_tools": ["research_search"],
            "reentry_step": "classifier"
        },
        "implementation_handler": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "implementation_instructions",
                    "template": (
                        "You are an implementation expert. The user's question has been classified as "
                        "implementation-oriented. Search the Vectara documentation corpus for relevant guides "
                        "and provide specific, actionable implementation guidance with API examples and "
                        "configuration details."
                    )
                }
            ],
            "output_parser": {"type": "default"},
            "allowed_tools": ["docs_search"],
            "reentry_step": "classifier"
        },
        "comparison_handler": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "comparison_instructions",
                    "template": (
                        "You are a comparison analyst. The user's question has been classified as a comparison. "
                        "Search both the research papers and documentation to find relevant information. "
                        "Provide a balanced comparison with evidence from both academic research and "
                        "practical documentation."
                    )
                }
            ],
            "output_parser": {"type": "default"},
            "allowed_tools": ["research_search", "docs_search"],
            "reentry_step": "classifier"
        }
    },
    "tool_configurations": {
        "research_search": {
            "type": "corpora_search",
            "query_configuration": {
                "search": {
                    "corpora": [{"corpus_key": research_corpus_key}],
                    "limit": 10
                }
            }
        },
        "docs_search": {
            "type": "corpora_search",
            "query_configuration": {
                "search": {
                    "corpora": [{"corpus_key": docs_corpus_key}],
                    "limit": 10
                }
            }
        }
    }
}

classifier_agent_key = delete_and_create_agent(classifier_agent_config, "Research Assistant Router")

Created agent 'Research Assistant Router' (key: agt_research_assistant_router_e91c)


### Step 4: Test Multi-Step Routing

Let's test with three different query types and observe the step transitions in the event stream.
All three tests share one session. Because every handler sets `reentry_step: "classifier"`, each new message is classified again before it is routed.

In [9]:
# One session for all three tests; reentry_step sends each new message back to the classifier
response = requests.post(
    f"{BASE_URL}/agents/{classifier_agent_key}/sessions",
    headers=headers,
    json={"name": f"Router Test {datetime.now().strftime('%Y%m%d-%H%M%S')}"}
)
response.raise_for_status()
router_session_key = response.json()["key"]

print("=" * 60)
print("TEST 1: Research query (expect: classifier -> research_handler)")
print("=" * 60)

query = "What are the key innovations in the original transformer architecture's attention mechanism?"
print(f"User: {query}\n")

result = chat_with_agent(classifier_agent_key, router_session_key, query, show_events=True)
print(f"Response:\n{result}")

TEST 1: Research query (expect: classifier -> research_handler)
User: What are the key innovations in the original transformer architecture's attention mechanism?




--- Agent Events ---
  Structured output (intent_classification): {"intent": "research", "reasoning": "The question asks about the key innovations in the attention mechanism of the original transformer architecture, which falls under understanding theoretical concep
  Step transition: classifier -> research_handler
  Tool call: research_search
  Tool response: research_search
  Agent output: The original transformer architecture, introduced by Vaswani et al. in their seminal paper "Attention is All You Need" (2017), brought several key innovations in the attention mechanism that have had ...
---

Response:
The original transformer architecture, introduced by Vaswani et al. in their seminal paper "Attention is All You Need" (2017), brought several key innovations in the attention mechanism that have had a profound impact on natural language processing and beyond. Here are the key innovations highlighted from the research:

1. **Self-Attention Mechanism**: The transformer model relies e

In [10]:
print("=" * 60)
print("TEST 2: Implementation query (expect: classifier -> implementation_handler)")
print("=" * 60)

query = "How do I configure hybrid search with reranking in Vectara's API?"
print(f"User: {query}\n")

result = chat_with_agent(classifier_agent_key, router_session_key, query, show_events=True)
print(f"Response:\n{result}")

TEST 2: Implementation query (expect: classifier -> implementation_handler)
User: How do I configure hybrid search with reranking in Vectara's API?




--- Agent Events ---
  Structured output (intent_classification): {"intent": "implementation", "reasoning": "The question is specifically about configuring a feature (hybrid search with reranking) using Vectara's API, which pertains to setting up or using Vectara's 
  Step transition: classifier -> implementation_handler
  Tool call: docs_search
  Tool response: docs_search
  Agent output: To configure hybrid search with reranking using Vectara's API, follow these steps:

1. **Understand Hybrid Search**: Hybrid search involves balancing neural search (using dense embeddings) with tradit...
---

Response:
To configure hybrid search with reranking using Vectara's API, follow these steps:

1. **Understand Hybrid Search**: Hybrid search involves balancing neural search (using dense embeddings) with traditional keyword-based search methods like BM25. This approach improves retrieval accuracy by leveraging the strengths of both retrieval methods.

2. **Implement Slingshot Reranking**: Reran

In [11]:
print("=" * 60)
print("TEST 3: Comparison query (expect: classifier -> comparison_handler)")
print("=" * 60)

query = "How does semantic search compare to keyword-based search for finding relevant documents?"
print(f"User: {query}\n")

result = chat_with_agent(classifier_agent_key, router_session_key, query, show_events=True)
print(f"Response:\n{result}")

TEST 3: Comparison query (expect: classifier -> comparison_handler)
User: How does semantic search compare to keyword-based search for finding relevant documents?




--- Agent Events ---
  Structured output (intent_classification): {"intent": "comparison", "reasoning": "The question seeks to understand the differences between semantic search and keyword-based search in the context of finding relevant documents, which involves co
  Step transition: classifier -> comparison_handler
  Tool call: research_search
  Tool call: docs_search
  Tool response: research_search
  Tool response: docs_search
  Agent output: The comparison between semantic search and keyword-based search is a significant topic in information retrieval, reflecting differing methodologies and advantages. Here's an overview:

### Semantic Se...
---

Response:
The comparison between semantic search and keyword-based search is a significant topic in information retrieval, reflecting differing methodologies and advantages. Here's an overview:

### Semantic Search

**Definition and Mechanism**:
- **Semantic search** utilizes advanced algorithms to understand the meaning behind search qu

## Cleanup (Optional)

Delete the agents created in this notebook:

In [12]:
for key, name in [(extraction_agent_key, "Research Entity Extractor"),
                  (classifier_agent_key, "Research Assistant Router")]:
    if key:
        response = requests.delete(f"{BASE_URL}/agents/{key}", headers=headers)
        if response.status_code == 204:
            print(f"Deleted agent: {name}")
        else:
            print(f"Error deleting {name}: {response.text}")

Deleted agent: Research Entity Extractor


Deleted agent: Research Assistant Router
